In [ ]:
import pandas as pd
from edgar import *
import os
from dotenv import load_dotenv

load_dotenv()
set_identity(os.getenv("EMAIL"))
print("All imported")

In [10]:
# Calculate Ebita
company = Company("AAPL")
financials = company.get_financials()

print(f"Company: {company.name}")
print(f"Revenue: ${financials.get_revenue():,.0f}")
print(f"Net Income: ${financials.get_net_income():,.0f}")

annual_reports = company.get_filings(form="10-K")

Company: Apple Inc.
Revenue: $416,161,000,000
Net Income: $112,010,000,000


In [11]:
inc = company.income_statement()
df_inc = inc.to_dataframe()
cashf = company.cashflow_statement()
df_cash = cashf.to_dataframe()

In [12]:
df_inc["label"]
row = df_inc.iloc[9]
operating_income = row.iloc[6]
print(operating_income)

a = df_inc[df_inc["label"].str.contains("Operating Income", na=False)]
b = a["FY 2025"]

133050000000.0


In [13]:
df_cash["label"]
row = df_cash.iloc[2]
dep_am = row.iloc[6]
print(row)

label          Depreciation, Depletion and Amortization
depth                                                 2
is_abstract                                       False
is_total                                          False
section                                            None
confidence                                     0.383686
FY 2025                                   11698000000.0
FY 2024                                   11445000000.0
FY 2023                                   11519000000.0
FY 2022                                   11104000000.0
Name: DepreciationDepletionAndAmortization, dtype: object


In [14]:
def ebitdas(c_name):

    # get the df of cashflow and income statement
    company = Company(c_name)
    financials = company.get_financials()
    inc = company.income_statement()
    df_inc = inc.to_dataframe()
    cashf = company.cashflow_statement()
    df_cash = cashf.to_dataframe()

    # Find the operating income
    year_cols = [c for c in df_inc.columns if "FY" in c]
    year = year_cols[0]
    print(year)

    a = df_inc.loc[df_inc.index.str.contains("OperatingIncomeLoss", na=False), year]
    if not a.empty:
        operating_income = int(a.iloc[0])
    else:
        operating_income = 0

    # Find the amortization
    a = df_cash.loc[
        df_cash.index.str.contains("DepreciationDepletionAndAmortization", na=False),
        year,
    ]
    if not a.empty:
        dep_am = a.iloc[0]
    else:
        a = df_cash.loc[
            df_cash.index.str.contains("DepreciationAndAmortization", na=False), year
        ]

        if not a.empty:
            dep_am = a.iloc[0]
        else:
            a = df_cash.loc[df_cash.index.str.contains("Depreciation", na=False), year]
            if not a.empty:
                dep_am = a.iloc[0]
            else:
                dep_am = 0
    if not a.empty and pd.notna(a.iloc[0]):
        dep_am = a.iloc[0]
    else:
        dep_am = 0

    ebitda = dep_am + operating_income
    print(
        f"{c_name}: Operating income:{operating_income / 1e6:.1f}M, Depresiation and amortization of {dep_am / 1e6:.1f}M  and {ebitda / 1e6:.1f}M of ebitda"
    )

    if operating_income == 0 or dep_am == 0 or year == "FY 2024":
        confidence = "low"
    else:
        confidence = "high"

    return {
        "operating_income": operating_income,
        "dep_am": dep_am,
        "ebitda": ebitda,
        "year": year,
        "confidence": confidence,
    }

In [15]:
tickers = ["NOVT", "MSA", "NOV", "TEL", "DKS", "TSCO", "ULBI", "WINA", "NSSC", "GNTX"]
for c in tickers:
    ebitdas(c)

FY 2025
NOVT: Operating income:94.0M, Depresiation and amortization of 61.9M  and 155.9M of ebitda
FY 2025
MSA: Operating income:371.8M, Depresiation and amortization of 71.6M  and 443.4M of ebitda
FY 2025
NOV: Operating income:494.0M, Depresiation and amortization of 355.0M  and 849.0M of ebitda
FY 2025
TEL: Operating income:3211.0M, Depresiation and amortization of 838.0M  and 4049.0M of ebitda
FY 2024
DKS: Operating income:1282.4M, Depresiation and amortization of 393.9M  and 1676.3M of ebitda
FY 2025
TSCO: Operating income:1467.4M, Depresiation and amortization of 494.0M  and 1961.4M of ebitda
FY 2024
ULBI: Operating income:10.0M, Depresiation and amortization of 0.0M  and 10.0M of ebitda
FY 2025
WINA: Operating income:54.6M, Depresiation and amortization of 0.7M  and 55.3M of ebitda
FY 2025
NSSC: Operating income:46.3M, Depresiation and amortization of 2.0M  and 48.2M of ebitda
FY 2025
GNTX: Operating income:473.9M, Depresiation and amortization of 104.0M  and 578.0M of ebitda


In [233]:
def funds_table(
    ebitda_results,
    entry_multiple=10,
    pct_debt=0.70,
    pct_senior=0.60,
    pct_sub=0.40,
    transaction_fee_pct=0.02,
    financing_fee_pct=0.035,
    mgmt_rollover_pct=0.10,
    text=False,
):

    ebitda = ebitda_results["ebitda"]

    # Enterprise Value
    TEV = ebitda * entry_multiple

    # Debt
    total_debt = TEV * pct_debt
    senior_debt = total_debt * pct_senior
    sub_debt = total_debt * pct_sub

    # Fees
    transaccion_fees = TEV * transaction_fee_pct
    financing_fees = total_debt * financing_fee_pct

    # Equity
    total_uses = TEV + transaccion_fees + financing_fees
    equity_requirement = total_uses - total_debt
    managment_rollover = equity_requirement * mgmt_rollover_pct
    sponsor_equity = equity_requirement - managment_rollover

    # Leverage
    total_debt_x = total_debt / ebitda
    senior_debt_x = senior_debt / ebitda

    # I would have store the data like this but AI organize it
    results = {
        "--USES-------------------": "",
        "TEV": TEV,
        "transaccion fees": transaccion_fees,
        "financing fees": financing_fees,
        "total uses": total_uses,
        "--SOURCES-----------------": "",
        "senior debt": senior_debt,
        "Sub/ HY debt": sub_debt,
        "total debt": total_debt,
        "managment rollover": managment_rollover,
        "sponsor equity": sponsor_equity,
        "Total Sources": total_debt + managment_rollover + sponsor_equity,
        "--CHECKS------------------": "",
        "Total Debt / EBITDA": f"{total_debt_x:.1f}x",
        "Senior Debt / EBITDA": f"{senior_debt_x:.1f}x",
        "Equity %": f"{(equity_requirement / total_uses):.1%}",
    }

    # AI did it for good visual representation
    if text == False:
        pass
    else:
        for k, v in results.items():
            if v == "":
                print(f"\n{k}")
            else:
                label = f"  {k:<30}"
                value = f"{v:>12,.0f}" if isinstance(v, float) else f"{v:>12}"
                print(label + value)

    return results

In [6]:
funds_table(100000000)


--USES-------------------
  TEV                             1000000000
  transaccion fees                20,000,000
  financing fees                  24,500,000
  total uses                    1,044,500,000

--SOURCES-----------------
  senior debt                    420,000,000
  Sub/ HY debt                   280,000,000
  total debt                     700,000,000
  managment rollover              34,450,000
  sponsor equity                 310,050,000
  Total Sources                 1,044,500,000

--CHECKS------------------
  Total Debt / EBITDA                   7.0x
  Senior Debt / EBITDA                  4.2x
  Equity %                             33.0%


{'--USES-------------------': '',
 'TEV': 1000000000,
 'transaccion fees': 20000000.0,
 'financing fees': 24500000.000000004,
 'total uses': 1044500000.0,
 '--SOURCES-----------------': '',
 'senior debt': 420000000.0,
 'Sub/ HY debt': 280000000.0,
 'total debt': 700000000.0,
 'managment rollover': 34450000.0,
 'sponsor equity': 310050000.0,
 'Total Sources': 1044500000.0,
 '--CHECKS------------------': '',
 'Total Debt / EBITDA': '7.0x',
 'Senior Debt / EBITDA': '4.2x',
 'Equity %': '33.0%'}

In [24]:
tickers = ["AAPL", "NKE", "CAT", "WMT", "XOM", "JPM", "UNH", "AMT", "LMT", "DECK"]
for i in tickers:
    company = Company(i)
    financials = company.get_financials()

    df_cash = company.cashflow_statement().to_dataframe()
    df_bal = company.balance_sheet().to_dataframe()
    df_inc = company.income_statement().to_dataframe()

    year_cols = [c for c in df_cash.columns if "FY" in c]
    year = year_cols[0]

    print(f" {i}")
    print("=== CASH FLOW TAGS ===")
    print(df_cash[year].dropna().to_string())

    print("\n=== BALANCE SHEET TAGS ===")
    print(df_bal[year].dropna().to_string())

    print("\n=== INCOME STATEMENT TAGS ===")
    print(df_inc[year].dropna().to_string())

 AAPL
=== CASH FLOW TAGS ===
concept
DepreciationDepletionAndAmortization                                                                              1.169800e+10
ShareBasedCompensation                                                                                            1.286300e+10
IncreaseDecreaseInAccountsReceivable                                                                              6.682000e+09
IncreaseDecreaseInInventories                                                                                    -1.400000e+09
IncreaseDecreaseInAccountsPayable                                                                                 9.020000e+08
NetCashProvidedByUsedInOperatingActivities                                                                        1.114820e+11
NetCashProvidedByUsedInInvestingActivities                                                                        1.519500e+10
PaymentsForProceedsFromOtherInvestingActivities                           

In [77]:
def fcf_data(c_name):
    company = Company(c_name)
    financials = company.get_financials()

    df_cash = company.cashflow_statement().to_dataframe()
    df_bal = company.balance_sheet().to_dataframe()
    df_inc = company.income_statement().to_dataframe()

    year_cols = [c for c in df_cash.columns if "FY" in c]
    year = year_cols[0]
    year_prev = year_cols[1]

    # --------Getting values---------
    CAPEX = 0
    for tag in ["PaymentsToAcquirePropertyPlantAndEquipment", "CapitalExpenditures"]:
        a = df_cash.loc[df_cash.index.str.contains(tag, na=False), year]
        if not a.empty and pd.notna(a.iloc[0]):
            CAPEX = a.iloc[0]
            break

    assets_m = df_bal.loc[df_bal.index == "AssetsCurrent"]
    liab_m = df_bal.loc[df_bal.index == "LiabilitiesCurrent"]
    NWC = (
        (assets_m[year].iloc[0] - liab_m[year].iloc[0])
        if (not assets_m.empty and not liab_m.empty)
        else None
    )
    NWC_1 = (
        (assets_m[year_prev].iloc[0] - liab_m[year_prev].iloc[0])
        if (not assets_m.empty and not liab_m.empty)
        else None
    )

    NWC_change = NWC - NWC_1

    tax_m = df_inc.loc[df_inc.index == "IncomeTaxExpenseBenefit"]
    pretax_m = df_inc.loc[
        df_inc.index.str.contains(
            "IncomeLossFromContinuingOperationsBeforeIncomeTaxes", na=False
        )
    ]
    tax_rate = (
        (tax_m[year].iloc[0] / pretax_m[year].iloc[0])
        if (not tax_m.empty and not pretax_m.empty and pretax_m[year].iloc[0] != 0)
        else None
    )

    if CAPEX == 0 or NWC is None:
        tax_rate = 0.25
        confidence = "low"
    else:
        confidence = "high"

    print(f"{c_name} | {year}")
    print(f"  CapEx:      {CAPEX / 1e6:.1f}M")
    print(
        f"  NWC:        {NWC / 1e6:.1f}M"
        if NWC is not None
        else "  NWC:      ⚠️ not found"
    )
    print(
        f"  NWC_change: {NWC_change / 1e6:.1f}M"
        if NWC_change is not None
        else "  NWC:      ⚠️ not found"
    )
    print(
        f"  Tax rate:   {tax_rate:.1%}"
        if tax_rate is not None
        else "  Tax rate: ⚠️ not found"
    )
    print(f"  Confidence: {confidence}  ")
    print("-----------------------------------------------")

    return {
        "CAPEX": CAPEX,
        "NWC": NWC,
        "NWC_change": NWC_change,
        "tax_rate": tax_rate,
        "year": year,
        "confidence": confidence,
    }

In [78]:
fcf_data("TSLA")

TSLA | FY 2025
  CapEx:      8527.0M
  NWC:        36928.0M
  NWC_change: 7389.0M
  Tax rate:   27.0%
  Confidence: high  
-----------------------------------------------


{'CAPEX': 8527000000.0,
 'NWC': 36928000000.0,
 'NWC_change': 7389000000.0,
 'tax_rate': 0.2696097006441834,
 'year': 'FY 2025',
 'confidence': 'high'}

In [147]:
def fcf_model(ebitdas_data, fcf_data, ebitda_growth=0.05, years=5, nwc_pct=0.02):

    ebitda = ebitdas_data["ebitda"]
    dep_am = ebitdas_data["dep_am"]
    tax_rate = fcf_data["tax_rate"] or 0.25
    capex = fcf_data["CAPEX"]
    nwc_change = fcf_data["NWC_change"]
    nwc_pct_historical = nwc_change / ebitda  # historical ratio
    nwc_pct_capped = min(nwc_pct_historical, nwc_pct)  # cap at parameter defau

    years_list = []
    ebitda_list = []
    ebit_list = []
    nopat_list = []
    fcf_list = []

    for year in range(1, years + 1):  # ← years + 1 so it goes 1 to 5 inclusive
        ebitda_yr = ebitda * (1 + ebitda_growth) ** (year - 1)
        ebit = ebitda_yr - dep_am
        tax = ebit * tax_rate
        nopat = ebit - tax
        fcf = nopat + dep_am - capex - (ebitda_yr * nwc_pct_capped)

        years_list.append(year)
        ebitda_list.append(round(ebitda_yr))
        ebit_list.append(round(ebit))
        nopat_list.append(round(nopat))
        fcf_list.append(round(fcf))

    return {
        "years_list": years_list,
        "ebitda_list": ebitda_list,
        "ebit_list": ebit_list,
        "nopat_list": nopat_list,
        "fcf_list": fcf_list,
    }


a = ebitdas("NOVT")
b = fcf_data("NOVT")

FY 2025
NOVT: Operating income:94.0M, Depresiation and amortization of 61.9M  and 155.9M of ebitda
NOVT | FY 2025
  CapEx:      15.6M
  NWC:        570.2M
  NWC_change: 304.4M
  Tax rate:   22.7%
  Confidence: high  
-----------------------------------------------


In [133]:
def debt_schedule(
    funds_results,
    fcf_results,
    senior_rate=0.07,
    sub_rate=0.10,
    amort_pct=0.05,
    sweep_pct=0.5,
    years=5,
):

    origianl_senior = funds_results["senior debt"]
    beginning_senior = origianl_senior
    original_sub = funds_results["Sub/ HY debt"]
    beginning_sub = original_sub
    fcf_list = fcf_results["fcf_list"]

    senior_beginning_list = []
    senior_ending_list = []
    senior_interest_list = []
    sub_ending_list = []
    sub_interest_list = []
    total_interest_list = []
    total_debt_list = []

    for i in range(years):
        mandatory_amort = -min(origianl_senior * amort_pct, beginning_senior)
        cash_after_amort = fcf_list[i] + mandatory_amort
        optional_sweep = -min(
            max(cash_after_amort, 0) * sweep_pct, beginning_senior + mandatory_amort
        )
        ending_senior = beginning_senior + mandatory_amort + optional_sweep
        interest_senior = ((beginning_senior + ending_senior) / 2) * senior_rate

        ending_sub = beginning_sub
        interest_sub = ((beginning_sub + ending_sub) / 2) * sub_rate

        beginning_senior = ending_senior
        beginning_sub = ending_sub

        senior_beginning_list.append(round(beginning_senior))
        senior_ending_list.append(round(ending_senior))
        senior_interest_list.append(round(interest_senior))
        sub_ending_list.append(round(ending_sub))
        sub_interest_list.append(round(interest_sub))
        total_interest_list.append(round(interest_senior + interest_sub))
        total_debt_list.append(round(ending_senior + ending_sub))

    return {
        "senior_beginning_list": senior_beginning_list,
        "senior_ending_list": senior_ending_list,
        "senior_interest_list": senior_interest_list,
        "sub_ending_list": sub_ending_list,
        "sub_interest_list": sub_interest_list,
        "total_interest_list": total_interest_list,
        "total_debt_list": total_debt_list,
    }

In [231]:
def returns(
    fcf_results, debt_results, funds_results, exit_multiple=9, years=5, text=False
):

    ebitda_last_year = fcf_results["ebitda_list"][-1]
    remaining_debt = debt_results["total_debt_list"][-1]
    sponsor_equity = funds_results["sponsor equity"]

    exit_tev = ebitda_last_year * exit_multiple
    equity_value = exit_tev - remaining_debt
    MoM = equity_value / sponsor_equity
    IRR = (MoM) ** (1 / years) - 1

    if text == False:
        pass
    else:
        if IRR > 0.2:
            print(f" IRR: {IRR:.1%} — acceptable")
        else:
            print(f" IRR: {IRR:.1%} — not acceptable")

        if MoM > 2.5:
            print(f" MoM: {MoM:.2f}x — acceptable")
        else:
            print(f" MoM: {MoM:.2f}x — not acceptable")

    return {"IRR": IRR, "MoM": MoM, "exit_tev": exit_tev, "equity_value": equity_value}

In [158]:
def evaluate_company(c_name, entry_multiple=10):
    try:
        a = ebitdas(c_name)
        b = fcf_data(c_name)
        c = fcf_model(a, b)
        d = funds_table(a, entry_multiple=entry_multiple)
        e = debt_schedule(d, c)
        f = returns(c, e, d)

        return {
            "ticker": c_name,
            "year": a["year"],
            "ebitda_M": round(a["ebitda"] / 1e6, 1),
            "capex_M": round(b["CAPEX"] / 1e6, 1),
            "nwc_change_M": round(b["NWC_change"] / 1e6, 1),
            "tax_rate": round(b["tax_rate"] * 100, 1),
            "TEV_M": round(d["TEV"] / 1e6, 1),
            "sponsor_equity_M": round(d["sponsor equity"] / 1e6, 1),
            "IRR": round(f["IRR"] * 100, 1),
            "MoM": round(f["MoM"], 2),
            "pass": f["IRR"] > 0.20,
            "confidence": a["confidence"],
        }
    except Exception as ex:
        print(f"⚠️ {c_name} failed: {ex}")
        return None

In [161]:
import time

tickers = [
    "MANH",
    "PRGS",
    "EPAC",
    "AMSF",
    "HURN",
    "MGRC",
    "ARIS",
    "NSSC",
    "ROAD",
    "CSWI",
]
for i in tickers:
    evaluate_company(i)
    time.sleep(5)

FY 2025
MANH: Operating income:279.8M, Depresiation and amortization of 6.3M  and 286.1M of ebitda
MANH | FY 2025
  CapEx:      15.5M
  NWC:        127.4M
  NWC_change: 24.5M
  Tax rate:   23.1%
  Confidence: high  
-----------------------------------------------

--USES-------------------
  TEV                           2,861,170,000
  transaccion fees                57,223,400
  financing fees                  70,098,665
  total uses                    2,988,492,065

--SOURCES-----------------
  senior debt                   1,201,691,400
  Sub/ HY debt                   801,127,600
  total debt                    2,002,819,000
  managment rollover              98,567,307
  sponsor equity                 887,105,759
  Total Sources                 2,988,492,065

--CHECKS------------------
  Total Debt / EBITDA                   7.0x
  Senior Debt / EBITDA                  4.2x
  Equity %                             33.0%
 
 IRR: 15.6% — not acceptable
 MoM: 2.07x — not acceptable
FY 

In [163]:
rows = []
for ticker in tickers:
    result = evaluate_company(ticker, entry_multiple=7)
    if result is not None:
        rows.append(result)
    time.sleep(5)

df = pd.DataFrame(rows).set_index("ticker")
df = df.sort_values("IRR", ascending=False)
print(df)

FY 2025
MANH: Operating income:279.8M, Depresiation and amortization of 6.3M  and 286.1M of ebitda
MANH | FY 2025
  CapEx:      15.5M
  NWC:        127.4M
  NWC_change: 24.5M
  Tax rate:   23.1%
  Confidence: high  
-----------------------------------------------

--USES-------------------
  TEV                           2,002,819,000
  transaccion fees                40,056,380
  financing fees                  49,069,066
  total uses                    2,091,944,446

--SOURCES-----------------
  senior debt                    841,183,980
  Sub/ HY debt                   560,789,320
  total debt                    1,401,973,300
  managment rollover              68,997,115
  sponsor equity                 620,974,031
  Total Sources                 2,091,944,446

--CHECKS------------------
  Total Debt / EBITDA                   4.9x
  Senior Debt / EBITDA                  2.9x
  Equity %                             33.0%
 
 IRR: 30.9% — acceptable
 MoM: 3.85x — acceptable
FY 2025
PRGS

In [235]:
def compare_entries_exits(c_name, Text=False):

    a = ebitdas(c_name)
    b = fcf_data(c_name)
    c = fcf_model(a, b)
    total = []

    for i in range(5, 11):
        rows = []
        for j in range(5, 11):
            d = funds_table(a, entry_multiple=j, text=Text)
            e = debt_schedule(d, c)
            f = returns(c, e, d, exit_multiple=i, text=Text)
            rows.append(round(f["IRR"] * 100, 1))

        total.append(rows)

    df = pd.DataFrame(
        total,
        columns=[f"Entry {x}" for x in range(5, 11)],
        index=[f"Exit {x}" for x in range(5, 11)],
    )

    return df

In [237]:
df = compare_entries_exits("MANH", Text=True)

FY 2025
MANH: Operating income:279.8M, Depresiation and amortization of 6.3M  and 286.1M of ebitda
MANH | FY 2025
  CapEx:      15.5M
  NWC:        127.4M
  NWC_change: 24.5M
  Tax rate:   23.1%
  Confidence: high  
-----------------------------------------------

--USES-------------------
  TEV                           1,430,585,000
  transaccion fees                28,611,700
  financing fees                  35,049,332
  total uses                    1,494,246,032

--SOURCES-----------------
  senior debt                    600,845,700
  Sub/ HY debt                   400,563,800
  total debt                    1,001,409,500
  managment rollover              49,283,653
  sponsor equity                 443,552,879
  Total Sources                 1,494,246,032

--CHECKS------------------
  Total Debt / EBITDA                   3.5x
  Senior Debt / EBITDA                  2.1x
  Equity %                             33.0%
 IRR: 24.7% — acceptable
 MoM: 3.02x — acceptable

--USES-------

In [225]:
def highlight_irr(val):
    if val >= 25:
        return "background-color: #d4edda"
    elif val >= 20:
        return "background-color: #fff3cd"
    else:
        return "background-color: #f8d7da"


df.style.format("{:.1f}%").applymap(highlight_irr)

/var/folders/hb/3snbzqyx6yd_96gxhv_h781w0000gn/T/ipykernel_98444/3335364318.py:9: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  df.style.format("{:.1f}%").applymap(highlight_irr)


,Entry 5,Entry 6,Entry 7,Entry 8,Entry 9,Entry 10
Exit 5,24.7%,17.4%,10.0%,2.8%,-4.7%,-12.9%
Exit 6,30.6%,23.6%,16.8%,10.4%,4.1%,-2.3%
Exit 7,35.6%,28.7%,22.2%,16.3%,10.7%,5.1%
Exit 8,40.0%,33.2%,26.9%,21.2%,15.9%,10.9%
Exit 9,43.8%,37.1%,30.9%,25.5%,20.4%,15.6%
Exit 10,47.3%,40.6%,34.5%,29.2%,24.3%,19.7%
